B18. Структурные аблации протокола: сколько вопросов и сколько пунктов.....


In [1]:
import sys, os, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/protocol_design')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
QPE = sweep_arms('questions_per_expert')
QIT = sweep_arms('questionnaire_items')
print()

def level_of(name):
    m = re.search('_(\\d+)$', name)
    return int(m.group(1)) if m else np.nan
SWEEP = {'вопросов на эксперта': {level_of(n): d for n, d in QPE.items()}, 'пунктов опросника': {level_of(n): d for n, d in QIT.items()}}
for k, v in SWEEP.items():
    print('%-24s уровни: %s' % (k, sorted(v)))


свип questions_per_expert: плеч 10, кейсов 60


свип questionnaire_items: плеч 18, кейсов 60

вопросов на эксперта     уровни: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
пунктов опросника        уровни: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [2]:
rows = []
for sweep, levels in SWEEP.items():
    for lv in sorted(levels):
        d = levels[lv]
        js_ = pd.to_numeric(d['Judge_score'], errors='coerce')
        ps = participants(d)
        votes = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in ps})
        nq = pd.to_numeric(d.get('organizer_after_r1_n_unified_questions'), errors='coerce')
        nsub = [c for c in d.columns if c.startswith('r1_n_subquestions__')]
        sub_mean = d[nsub].apply(pd.to_numeric, errors='coerce').mean(axis=1).mean() if nsub else np.nan
        rr = pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce')
        rows.append({'свип': sweep, 'уровень': lv, 'кейсов': len(d), 'Judge': float(js_.mean()), 'SE': se_clt(js_.dropna()), 'полнота голосов': float(votes.notna().all(axis=1).mean()), 'факт. подвопросов': sub_mean, 'факт. пунктов опросника': float(nq.mean()) if nq is not None else np.nan, 'ср. раундов': float(rr.mean()) if rr is not None else np.nan})
B = pd.DataFrame(rows)
show(B.set_index(['свип', 'уровень']), 'Свипы структуры протокола')



Свипы структуры протокола
------------------------------------------------------------
                              кейсов  Judge     SE  полнота голосов  факт. подвопросов  факт. пунктов опросника  ср. раундов
свип                 уровень                                                                                                
вопросов на эксперта 3            60 0.8214 0.0153           0.9667             2.9667                   7.2167       1.8833
                     4            60 0.8040 0.0158           0.9333             3.9111                   7.2667       1.7833
                     5            60 0.8162 0.0145           0.9167             4.8611                   7.2000       1.6333
                     6            60 0.7809 0.0184           0.9167             5.8333                   7.0000       1.7667
                     7            60 0.8311 0.0146           0.9833             6.9611                   7.7000       1.7333
                     8            60 

In [3]:
for sweep, col in [('вопросов на эксперта', 'факт. подвопросов'), ('пунктов опросника', 'факт. пунктов опросника')]:
    s = B[B['свип'] == sweep][['уровень', col]].dropna()
    if len(s) >= 4:
        r, p = stats.spearmanr(s['уровень'], s[col])
        print('%-24s заданный уровень -> %-26s rho = %+.3f  p = %s' % (sweep, col, r, fmt_p(p)))
print()
print('Высокая положительная корреляция = параметр действительно управляет')
print('тем, чем должен. Низкая = свип не сработал, и выводы делать нельзя.')


вопросов на эксперта     заданный уровень -> факт. подвопросов          rho = +1.000  p = 6.6e-64
пунктов опросника        заданный уровень -> факт. пунктов опросника    rho = +0.891  p = 7.0e-07

Высокая положительная корреляция = параметр действительно управляет
тем, чем должен. Низкая = свип не сработал, и выводы делать нельзя.


In [4]:
def curve_stats(levels, label):
    parts = []
    for lv, d in levels.items():
        s = pd.to_numeric(d['Judge_score'], errors='coerce').dropna()
        parts.append(pd.DataFrame({'y': s.to_numpy(), 'lv': lv, 'case': s.index}))
    L = pd.concat(parts, ignore_index=True)
    L['lv_c'] = L['lv'] - L['lv'].mean()
    L['lv2'] = L['lv_c'] ** 2
    CD = pd.get_dummies(L['case'], prefix='c', drop_first=True).astype(float)
    g = L['case'].astype('category').cat.codes
    m1 = sm.OLS(L['y'], sm.add_constant(pd.concat([L[['lv_c']], CD], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g})
    m2 = sm.OLS(L['y'], sm.add_constant(pd.concat([L[['lv_c', 'lv2']], CD], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g})
    means = L.groupby('lv')['y'].mean()
    rho, prho = stats.spearmanr(means.index, means.to_numpy())
    b2 = m2.params.get('lv2', 0.0)
    opt = -m2.params['lv_c'] / (2 * b2) + L['lv'].mean() if b2 < 0 else np.nan
    return ({'свип': label, 'наблюдений': len(L), 'уровней': L['lv'].nunique(), 'линейный коэф': m1.params['lv_c'], 'p линейн.': m1.pvalues['lv_c'], 'квадратичный коэф': b2, 'p квадр.': m2.pvalues.get('lv2', np.nan), 'оптимум': opt, 'диапазон': '%d-%d' % (L['lv'].min(), L['lv'].max()), 'rho (по средним)': rho, 'p rho': prho}, L)
rows, LONG = ([], {})
for sweep, levels in SWEEP.items():
    r, L = curve_stats(levels, sweep)
    rows.append(r)
    LONG[sweep] = L
CS = pd.DataFrame(rows).set_index('свип')
show(CS, 'Тренды по свипам (регрессия с фиксированным эффектом кейса)')
print()
print('«линейный коэф» — изменение Judge_score на один шаг параметра.')
print('Умножьте на размах диапазона, чтобы получить эффект от края до края.')
for _, r in CS.reset_index().iterrows():
    lo, hi = [int(x) for x in r['диапазон'].split('-')]
    print('  %-24s эффект от %d до %d: %+.4f' % (r['свип'], lo, hi, r['линейный коэф'] * (hi - lo)))



Тренды по свипам (регрессия с фиксированным эффектом кейса)
------------------------------------------------------------
                      наблюдений  уровней  линейный коэф  p линейн.  квадратичный коэф  p квадр.  оптимум диапазон  rho (по средним)  p rho
свип                                                                                                                                       
вопросов на эксперта         600       10        -0.0007     0.6282            -0.0007    0.1950   7.0304     3-12           -0.1273 0.7261
пунктов опросника           1080       18        -0.0008     0.1421             0.0001    0.4722      NaN     3-20           -0.3313 0.1793

«линейный коэф» — изменение Judge_score на один шаг параметра.
Умножьте на размах диапазона, чтобы получить эффект от края до края.
  вопросов на эксперта     эффект от 3 до 12: -0.0062
  пунктов опросника        эффект от 3 до 20: -0.0136


In [5]:
for sweep in SWEEP:
    sub = B[B['свип'] == sweep].set_index('уровень')[['Judge', 'SE', 'ср. раундов']]
    show(sub, 'Кривая: %s' % sweep)



Кривая: вопросов на эксперта
------------------------------------------------------------
         Judge     SE  ср. раундов
уровень                           
3       0.8214 0.0153       1.8833
4       0.8040 0.0158       1.7833
5       0.8162 0.0145       1.6333
6       0.7809 0.0184       1.7667
7       0.8311 0.0146       1.7333
8       0.8206 0.0151       1.7167
9       0.8304 0.0156       1.8333
10      0.8183 0.0143       1.8167
11      0.8094 0.0166       1.7333
12      0.7880 0.0185       1.7500

Кривая: пунктов опросника
------------------------------------------------------------
         Judge     SE  ср. раундов
уровень                           
3       0.8386 0.0136       1.7167
4       0.8119 0.0141       1.7167
5       0.8140 0.0143       1.7000
6       0.8201 0.0129       1.8000
7       0.8147 0.0150       1.6833
8       0.8271 0.0155       1.7333
9       0.8237 0.0135       1.7500
10      0.8405 0.0147       1.7500
11      0.8100 0.0159       1.7500
12      0.8214 0

In [6]:
def tost(d, margin):
    n = len(d)
    m, se = (float(d.mean()), se_clt(d))
    return (max(float(stats.t.sf((m + margin) / se, n - 1)), float(stats.t.cdf((m - margin) / se, n - 1))), m)
rows = []
for sweep, levels in SWEEP.items():
    top = max(levels)
    ref = pd.to_numeric(levels[top]['Judge_score'], errors='coerce')
    for lv in sorted(levels):
        if lv == top:
            continue
        s = pd.to_numeric(levels[lv]['Judge_score'], errors='coerce')
        ix = s.dropna().index.intersection(ref.dropna().index)
        d = s.loc[ix] - ref.loc[ix]
        p05, m = tost(d, 0.05)
        rows.append({'свип': sweep, 'уровень': lv, 'против': top, 'Δ': m, 'p TOST ±0.05': p05, 'вывод': 'эквивалентно' if p05 < 0.05 else 'не доказано'})
E = pd.DataFrame(rows)
show(E.set_index(['свип', 'уровень']), 'Можно ли уменьшить параметр без потери качества')
print()
print('Сводка: сколько уровней доказанно эквивалентны максимальному')
show(E.groupby('свип')['вывод'].value_counts().unstack(fill_value=0), None)



Можно ли уменьшить параметр без потери качества
------------------------------------------------------------
                              против       Δ  p TOST ±0.05         вывод
свип                 уровень                                            
вопросов на эксперта 3            12  0.0333        0.1547   не доказано
                     4            12  0.0159        0.0342  эквивалентно
                     5            12  0.0281        0.0933   не доказано
                     6            12 -0.0071        0.0193  эквивалентно
                     7            12  0.0431        0.3573   не доказано
                     8            12  0.0326        0.1700   не доказано
                     9            12  0.0424        0.3121   не доказано
                     10           12  0.0303        0.1313   не доказано
                     11           12  0.0213        0.0506   не доказано
пунктов опросника    3            20  0.0220        0.0227  эквивалентно
              

In [7]:
rows = []
for sweep, levels in SWEEP.items():
    for lv in sorted(levels):
        d = levels[lv]
        tot = [c for c in d.columns if re.match('r[123]_total_tokens__', c)]
        org = [c for c in ['organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens'] if c in d.columns]
        gen = d[tot].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean() if tot else np.nan
        agg = d[org].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean() if org else np.nan
        q = float(pd.to_numeric(d['Judge_score'], errors='coerce').mean())
        rows.append({'свип': sweep, 'уровень': lv, 'S генерация': gen, 'V агрегация': agg, 'total': gen + agg, 'Judge': q, 'Judge на 10k': q / ((gen + agg) / 10000.0) if gen + agg else np.nan})
C = pd.DataFrame(rows)
show(C.set_index(['свип', 'уровень']), 'Стоимость по уровням')
print()
for sweep in SWEEP:
    s = C[C['свип'] == sweep]
    r, p = stats.spearmanr(s['уровень'], s['total'])
    lo, hi = (s['total'].iloc[0], s['total'].iloc[-1])
    print('%-24s стоимость: rho с уровнем = %+.3f (p = %s), рост %.0f -> %.0f токенов' % (sweep, r, fmt_p(p), lo, hi))



Стоимость по уровням
------------------------------------------------------------
                              S генерация  V агрегация      total  Judge  Judge на 10k
свип                 уровень                                                          
вопросов на эксперта 3         14682.6000   10639.1167 25321.7167 0.8214        0.3244
                     4         13778.3000   10206.2667 23984.5667 0.8040        0.3352
                     5         12344.6500    9811.4333 22156.0833 0.8162        0.3684
                     6         13945.2833   10509.1000 24454.3833 0.7809        0.3193
                     7         13831.0833   11000.0500 24831.1333 0.8311        0.3347
                     8         13710.5333   11189.1833 24899.7167 0.8206        0.3296
                     9         15959.9333   12593.9500 28553.8833 0.8304        0.2908
                     10        15327.1000   12186.6000 27513.7000 0.8183        0.2974
                     11        15309.6667   127

In [8]:
rows = []
for sweep in SWEEP:
    eq = E[(E['свип'] == sweep) & (E['вывод'] == 'эквивалентно')]
    if not len(eq):
        rows.append({'свип': sweep, 'рекомендуемый уровень': 'нет доказанно эквивалентных', 'экономия токенов, %': np.nan})
        continue
    best = int(eq['уровень'].min())
    cs = C[C['свип'] == sweep].set_index('уровень')['total']
    top = int(max(SWEEP[sweep]))
    rows.append({'свип': sweep, 'рекомендуемый уровень': best, 'текущий максимум': top, 'токенов при рекоменд.': cs.get(best, np.nan), 'токенов при максимуме': cs.get(top, np.nan), 'экономия токенов, %': 100 * (1 - cs.get(best, np.nan) / cs.get(top, np.nan))})
show(pd.DataFrame(rows).set_index('свип'), 'Где можно сэкономить')



Где можно сэкономить
------------------------------------------------------------
                      рекомендуемый уровень  текущий максимум  токенов при рекоменд.  токенов при максимуме  экономия токенов, %
свип                                                                                                                            
вопросов на эксперта                      4                12             23984.5667             28080.1000              14.5852
пунктов опросника                         3                20             23088.6500             18408.9333             -25.4209


In [9]:
save(B, 'questions_and_criteria_ablation_sweeps.csv')
save(CS.reset_index(), 'questions_and_criteria_ablation_trends.csv')
save(E, 'questions_and_criteria_ablation_tost.csv')
save(C, 'questions_and_criteria_ablation_cost.csv')


сохранено: out/questions_and_criteria_ablation_sweeps.csv  (28 строк)
сохранено: out/questions_and_criteria_ablation_trends.csv  (2 строк)
сохранено: out/questions_and_criteria_ablation_tost.csv  (26 строк)
сохранено: out/questions_and_criteria_ablation_cost.csv  (28 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\protocol_design\\results\\questions_and_criteria_ablation_cost.csv'